In [35]:

import pandas as pd

import calendar
from datetime import datetime
from datetime import timedelta
from cronsim import CronSim

from datetime import date

today = date.today()
import argparse
import json
import PySimpleGUI as sg

today = date.today()
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)  # Set to None for no limit
# Or set a specific width

In [13]:
crons = []
def find_all(a_str, sub):
    start = 0
    while True:
        start = a_str.find(sub, start)
        if start == -1: return
        yield start
        start += len(sub) # use start += 1 to find overlapping matches
         
with open("/home/joe/bic_etl/general/cron/cron_file","r") as fin:
    for line in fin:
        if line[0:1] != "#"  and len(line) > 5:
           #print(line0)
#           print(line)
           crons.append(line.rstrip())
        line0=line
print(f"{len(crons)} Cron Jobs Found")

crons_all = []
ncrons=0
descriptions = []
for line in crons:
    line = line.strip(" ")
    spl = line.split(" ")
    crn=""
#    print(line)
    mm = line.find("node")
    nj = line.find("java")
    npyth = line.find("python")
    nt = line.find("-t")
    np = line.find("-p")
    if (np > 0):
        a = list(find_all(line[np:],'\"'))
        # if len(a) > 0:
        #   p=line[np+a[0]:np+a[1]+1]
        # else:
        ss = line[np:].split()
        p=ss[1]
    elif mm > 0:
        pp = spl[6].split("/")
        p= pp[-1]
        display("NODE ",p)
        
        
    elif nj > 0:  # java line
        pp = spl[7].split("/")
        p= pp[-1]
        display("JAVA ",p)
       
    elif npyth > 0:  # java line
        pp = spl[10].split("/")
        p= pp[-1]
        display("PYTHON ",p)
       
    else:
            p=""
            display("Find Program")
            display(line)
        
    if (nt > 0):
        ngt = line.find(">>")
        a = list(find_all(line[nt:ngt],'\"'))
        if len(a) > 0:
          t=line[nt+a[0]:nt+a[1]+1]
        else:
          ss = line[nt:].split()
          t=ss[1]
    else:
        t=""
            

    for val in spl[:5]:
        crn+= f"{val} "
    crn = crn.rstrip()
    if spl[5] == "node":
        pg = spl[6]
    else:
        pg=""
    print("PROGRAM",p)
    if len(p) > 0:  #  There is a program listed
      
        pgs=pg.split("/")
       
    #    print(line)
        mo = 1
        yr=today.year
        mo=today.month
        dy=today.day
        
#        date = datetime.strptime(f"{yr}-{mo}-01","%Y-%m-%d")
        date = datetime.strptime(f"{yr}-{mo}-{dy}","%Y-%m-%d")
        print("DATE  ",date)
        
        try:
            it = CronSim(crn,date)
            tmp={}
            tmp["line"]=line
            tmp["desc"] = it.explain()
         #   print("ME ",yr,mo,it.explain())
            descriptions.append(tmp)
            a = next(it) 
            cdate = datetime.strptime(f"{a.year}-{a.month}-{a.day}","%Y-%m-%d")
            ndays = (cdate-date).days
            print("Days ",ndays,a)
            while  ndays < 32:
         #       print(a.month,a.day,a.hour,a.minute,a.hour+a.minute/60)
         #       print("DAY ",calendar.day_name[a.weekday()])
          #      print(f"start:{a}   end:{a+timedelta(days=1)}")
                d = dict(Day=calendar.day_name[a.weekday()],Cron=line,T=t,TM=f"{a.hour}:{a.minute}",Task=p,Details=t,Program=pg,Start=a,End=a+timedelta(days=1),Time=a.hour+a.minute/60)
           #     crons_all.append(d)
                cdate = datetime.strptime(f"{a.year}-{a.month}-{a.day}","%Y-%m-%d")
                print(f"DD {crn}  {a.year}-{a.month}-{a.day}-{a.hour}-{a.minute}")
                ndays = (cdate-date).days
                if ndays <= 31:
                     crons_all.append(d)
               # print(date,cdate,ndays)
            # print(f"pg: {pg} P:{p} T:{t}")    
                a = next(it)
            ncrons+=1 
        except Exception as err:
            print("Count not Process",crn)
            print(err)
            print(line)
    else:  # Not a node runnning a cim dataset... must be java or python
        print(line)
    #  print("--------")

print(f"{len(crons_all)}  Crons successfully mapped to time ranges")

'NODE '

'pull_and_setup.js'

'JAVA '

'DataSync-1.8.2.jar'

'PYTHON '

'metadata_updater.py'

'NODE '

'cleanup.js'

In [14]:
hist={}
for dct in crons_all:
    cron = dct['Cron']
    if cron not in hist:
       hist[cron]=[]
    hist[cron].append(dct)
    

In [ ]:
hist

In [15]:
mapping = {32: "Daily",30:"Daily",4:"Weekly",5:"Weekly",1:"Monthly"}
mapped={}
counts={}                                                          
for cron,vals in hist.items():
    count=len(vals)
    if count > 30:
        print(count,cron)
    try:
        mapped[cron]=mapping[count]
        if count not in counts:
            counts[count]=0
        counts[count]+=1
    except:
        print("Bad ",count)
        for v in vals:
            print(v['Start'])

In [16]:
for cron in crons_all:
    cron['Schedule'] = mapped[cron['Cron']]

In [17]:
crons_all

[{'Day': 'Monday',
  'Cron': '50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> /usr/local/cim/bic_etl/general/logs/cron.log',
  'T': '',
  'TM': '2:50',
  'Task': 'pull_and_setup.js',
  'Details': '',
  'Program': '/usr/local/cim/bic_etl/general/scripts/pull_and_setup.js',
  'Start': datetime.datetime(2024, 12, 9, 2, 50),
  'End': datetime.datetime(2024, 12, 10, 2, 50),
  'Time': 2.8333333333333335,
  'Schedule': 'Daily'},
 {'Day': 'Tuesday',
  'Cron': '50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> /usr/local/cim/bic_etl/general/logs/cron.log',
  'T': '',
  'TM': '2:50',
  'Task': 'pull_and_setup.js',
  'Details': '',
  'Program': '/usr/local/cim/bic_etl/general/scripts/pull_and_setup.js',
  'Start': datetime.datetime(2024, 12, 10, 2, 50),
  'End': datetime.datetime(2024, 12, 11, 2, 50),
  'Time': 2.8333333333333335,
  'Schedule': 'Daily'},
 {'Day': 'Wednesday',
  'Cron': '50 2 * * * node /usr/local/cim/bic_etl/general/scripts/

In [18]:
info=[]
done = {}
for cron in crons_all:
    c = cron['Cron']
    if c not in done:
        done[c]={}
    day = cron['Day']
    if day not in done[c]: 
        info.append([cron['Day'],cron['TM'],cron['Schedule'],cron['Task'],cron['T'],cron['Cron'],cron['Time']])
        done[c][day]=1


In [19]:
day = datetime.now().strftime('%A')
display(day)

def getDay(day):

    tod={}
    for cron in crons_all:
        display(cron)
        if cron['Day'] == day:
           tod[cron['Cron']]=1
          
    keys = ["node","PATH","/usr/bin"]
    tod2 = tod.copy()
    toRun = []
    for string in tod:
        hit=0
        for key in keys:
            if string.find(key) > -1:
                st = string.find(key)
                string=string[st:]
                ed = string.find("2>>")
                hit=1
                if ed > -1:
                    toRun.append(string[:ed])
                else:
                    print("ROH ROH ",string)
        if hit == 0:
            print(string)
       
    
    return toRun

toRun = getDay(day)

'Monday'

{'Day': 'Monday',
 'Cron': '50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> /usr/local/cim/bic_etl/general/logs/cron.log',
 'T': '',
 'TM': '2:50',
 'Task': 'pull_and_setup.js',
 'Details': '',
 'Program': '/usr/local/cim/bic_etl/general/scripts/pull_and_setup.js',
 'Start': datetime.datetime(2024, 12, 9, 2, 50),
 'End': datetime.datetime(2024, 12, 10, 2, 50),
 'Time': 2.8333333333333335,
 'Schedule': 'Daily'}

{'Day': 'Tuesday',
 'Cron': '50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> /usr/local/cim/bic_etl/general/logs/cron.log',
 'T': '',
 'TM': '2:50',
 'Task': 'pull_and_setup.js',
 'Details': '',
 'Program': '/usr/local/cim/bic_etl/general/scripts/pull_and_setup.js',
 'Start': datetime.datetime(2024, 12, 10, 2, 50),
 'End': datetime.datetime(2024, 12, 11, 2, 50),
 'Time': 2.8333333333333335,
 'Schedule': 'Daily'}

{'Day': 'Wednesday',
 'Cron': '50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> /usr/local/cim/bic_etl/general/logs/cron.log',
 'T': '',
 'TM': '2:50',
 'Task': 'pull_and_setup.js',
 'Details': '',
 'Program': '/usr/local/cim/bic_etl/general/scripts/pull_and_setup.js',
 'Start': datetime.datetime(2024, 12, 11, 2, 50),
 'End': datetime.datetime(2024, 12, 12, 2, 50),
 'Time': 2.8333333333333335,
 'Schedule': 'Daily'}

{'Day': 'Thursday',
 'Cron': '50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> /usr/local/cim/bic_etl/general/logs/cron.log',
 'T': '',
 'TM': '2:50',
 'Task': 'pull_and_setup.js',
 'Details': '',
 'Program': '/usr/local/cim/bic_etl/general/scripts/pull_and_setup.js',
 'Start': datetime.datetime(2024, 12, 12, 2, 50),
 'End': datetime.datetime(2024, 12, 13, 2, 50),
 'Time': 2.8333333333333335,
 'Schedule': 'Daily'}

...

{'Day': 'Sunday',
 'Cron': '47 14 * * * node /usr/local/cim/bic_etl/general/scripts/cleanup.js 2>> /usr/local/cim/bic_etl/general/logs/cron.log',
 'T': '',
 'TM': '14:47',
 'Task': 'cleanup.js',
 'Details': '',
 'Program': '/usr/local/cim/bic_etl/general/scripts/cleanup.js',
 'Start': datetime.datetime(2025, 1, 5, 14, 47),
 'End': datetime.datetime(2025, 1, 6, 14, 47),
 'Time': 14.783333333333333,
 'Schedule': 'Daily'}

{'Day': 'Monday',
 'Cron': '47 14 * * * node /usr/local/cim/bic_etl/general/scripts/cleanup.js 2>> /usr/local/cim/bic_etl/general/logs/cron.log',
 'T': '',
 'TM': '14:47',
 'Task': 'cleanup.js',
 'Details': '',
 'Program': '/usr/local/cim/bic_etl/general/scripts/cleanup.js',
 'Start': datetime.datetime(2025, 1, 6, 14, 47),
 'End': datetime.datetime(2025, 1, 7, 14, 47),
 'Time': 14.783333333333333,
 'Schedule': 'Daily'}

{'Day': 'Tuesday',
 'Cron': '47 14 * * * node /usr/local/cim/bic_etl/general/scripts/cleanup.js 2>> /usr/local/cim/bic_etl/general/logs/cron.log',
 'T': '',
 'TM': '14:47',
 'Task': 'cleanup.js',
 'Details': '',
 'Program': '/usr/local/cim/bic_etl/general/scripts/cleanup.js',
 'Start': datetime.datetime(2025, 1, 7, 14, 47),
 'End': datetime.datetime(2025, 1, 8, 14, 47),
 'Time': 14.783333333333333,
 'Schedule': 'Daily'}

{'Day': 'Wednesday',
 'Cron': '47 14 * * * node /usr/local/cim/bic_etl/general/scripts/cleanup.js 2>> /usr/local/cim/bic_etl/general/logs/cron.log',
 'T': '',
 'TM': '14:47',
 'Task': 'cleanup.js',
 'Details': '',
 'Program': '/usr/local/cim/bic_etl/general/scripts/cleanup.js',
 'Start': datetime.datetime(2025, 1, 8, 14, 47),
 'End': datetime.datetime(2025, 1, 9, 14, 47),
 'Time': 14.783333333333333,
 'Schedule': 'Daily'}

{'Day': 'Thursday',
 'Cron': '47 14 * * * node /usr/local/cim/bic_etl/general/scripts/cleanup.js 2>> /usr/local/cim/bic_etl/general/logs/cron.log',
 'T': '',
 'TM': '14:47',
 'Task': 'cleanup.js',
 'Details': '',
 'Program': '/usr/local/cim/bic_etl/general/scripts/cleanup.js',
 'Start': datetime.datetime(2025, 1, 9, 14, 47),
 'End': datetime.datetime(2025, 1, 10, 14, 47),
 'Time': 14.783333333333333,
 'Schedule': 'Daily'}

In [27]:
toRun

['node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js ',
 '/usr/bin/java -jar /usr/local/cim/bic_etl/general/datasync/DataSync-1.8.2.jar -t LoadPreferences -c /usr/local/cim/bic_etl/general/datasync/config.json ',
 'PATH=$PATH:/home/giddensm/.local/bin PIPENV_PIPFILE=/usr/local/cim/bic_etl/general/metadata_updater/scripts/Pipfile pipenv run python /usr/local/cim/bic_etl/general/metadata_updater/scripts/metadata_updater.py -a ',
 'node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p boulder ',
 'node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p catalog ',
 'node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdos/business/nonprofit -t "Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado" -d /usr/local/cim/bic_etl/general/datasync/DataSync-1.9.1.jar ',
 'node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdos/business/business -t "Business Entities in Colorado" ',
 'no

In [10]:
def showInfo(day):
    display("Show Day ",day)
    data = []
    for vals in info:
        if vals[0] == day:
            data.append(vals)
    data = sorted(data, key=lambda x: x[-1]) 
 #   data = data[:][:-1]
    display(data)
    header=["Day","Time","Schedule","Group","Dataset","Cron"]
    layout = [
              [sg.Text(f"Detailed Info for {day}",font='Courier 30 bold ')],
              [sg.Button('Quit',font='Courier 15 bold')],
              [sg.Table(values=data,
               background_color='green',auto_size_columns=True,enable_events=True,
               justification='center',alternating_row_color='brown',size=(100,30),
               key='-TABLE-', headings = header,vertical_scroll_only = False)]
        ]

    sg.theme('Dark Green 5')
    window2 = sg.Window('ETL', layout,finalize=True,resizable=True,background_color="#FFD700")
    

In [ ]:
days = ["Sunday","Monday","Tuesday","Wednesday","Thursday","Friday","Saturday"]
layout = [
              [sg.Text("BIC Cron Finder",font='Courier 30 bold ')],
              [sg.Combo(days,enable_events=True,key="-DAYOFWEEK-",font='Courier 20 bold ')],
              [sg.Output(font='Courier 15 bold ',background_color="lightblue",key="-OUTPUT-",size=[120,20])],
              [sg.Button('Exit',font='Courier 15 bold')]
        ]

sg.theme('Dark Green 5')
window2 = sg.Window('ETL', layout,finalize=True,resizable=True,background_color="#FFD700")


try: 
    while True:

     #   event, values = window2.read()
        wid, event, values = sg.read_all_windows()
        print("\n\n-----------------------------------\n")
        print('event: ',event)
        print("VALS ",values)
        print(wid.Title)
        print(wid)

        if event == sg.WIN_CLOSED or event == "Exit":
            window2.close()
            break
        elif event == sg.WIN_CLOSED or event == "Quit":
            wid.close()
        elif event == "-DAYOFWEEK-":
            day = values['-DAYOFWEEK-']
            toRun = getDay(day)

            # string=""
            # for crn in toRun:
            #     string+= crn + "\n"

            wid["-OUTPUT-"].update("") 
            # wid["-OUTPUT-"].update(string) 
            for nn,crn in enumerate(toRun):
                if nn%2 == 0:
                    color="red"
                else:
                    color="green"
                wid['-OUTPUT-'].print(f"{crn}\n",text_color=color)
                
            showInfo(day)
            
              
            
            
except Exception as err:
    display(err)

In [20]:
info

[['Monday',
  '2:50',
  'Daily',
  'pull_and_setup.js',
  '',
  '50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> /usr/local/cim/bic_etl/general/logs/cron.log',
  2.8333333333333335],
 ['Tuesday',
  '2:50',
  'Daily',
  'pull_and_setup.js',
  '',
  '50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> /usr/local/cim/bic_etl/general/logs/cron.log',
  2.8333333333333335],
 ['Wednesday',
  '2:50',
  'Daily',
  'pull_and_setup.js',
  '',
  '50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> /usr/local/cim/bic_etl/general/logs/cron.log',
  2.8333333333333335],
 ['Thursday',
  '2:50',
  'Daily',
  'pull_and_setup.js',
  '',
  '50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> /usr/local/cim/bic_etl/general/logs/cron.log',
  2.8333333333333335],
 ['Friday',
  '2:50',
  'Daily',
  'pull_and_setup.js',
  '',
  '50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> /usr

In [29]:
df=pd.DataFrame(info,columns=["Day","Time","Schedule","Program","Unknow","Cron","Cron Orig"])

In [30]:
df.head()


,Day,Time,Schedule,Program,Unknow,Cron,Cron Orig
0,Monday,2:50,Daily,pull_and_setup.js,,50 2 * * * node /usr/local/cim/bic_etl/general...,2.833333
1,Tuesday,2:50,Daily,pull_and_setup.js,,50 2 * * * node /usr/local/cim/bic_etl/general...,2.833333
2,Wednesday,2:50,Daily,pull_and_setup.js,,50 2 * * * node /usr/local/cim/bic_etl/general...,2.833333
3,Thursday,2:50,Daily,pull_and_setup.js,,50 2 * * * node /usr/local/cim/bic_etl/general...,2.833333
4,Friday,2:50,Daily,pull_and_setup.js,,50 2 * * * node /usr/local/cim/bic_etl/general...,2.833333


In [36]:
df.loc[df["Schedule"] == "Monthly",["Schedule","Program","Unknow","Cron"]]

,Schedule,Program,Unknow,Cron
66,Monthly,cdos/business/business,"""Master List in Colorado""","30 5 4 * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdos/business/business -t ""Master List in Colorado"" 2>> /usr/local/cim/bic_etl/general/logs/cron.log"
76,Monthly,cdot/transportation_road_attributes,,10 4 4 * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdot/transportation_road_attributes -d /usr/local/cim/bic_etl/general/datasync/DataSync-1.9.8-jar-with-dependencies.jar 2>> /usr/local/cim/bic_etl/general/logs/cron.log
77,Monthly,cdot/transportation_infrastructure,,30 4 4 * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdot/transportation_infrastructure 2>> /usr/local/cim/bic_etl/general/logs/cron.log
78,Monthly,cdot/natural_resources,,50 4 4 * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdot/natural_resources 2>> /usr/local/cim/bic_etl/general/logs/cron.log
80,Monthly,dola/boundaries,,0 4 4 * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p dola/boundaries 2>> /usr/local/cim/bic_etl/general/logs/cron.log
81,Monthly,dola/special_districts,,20 5 4 * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p dola/special_districts 2>> /usr/local/cim/bic_etl/general/logs/cron.log
82,Monthly,dola/demographics,,30 4 1 * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p dola/demographics 2>> /usr/local/cim/bic_etl/general/logs/cron.log
85,Monthly,cdor/retail_reports,,30 4 10 * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdor/retail_reports 2>> /usr/local/cim/bic_etl/general/logs/cron.log


In [37]:
3/7

0.42857142857142855